# DEPRECATED ENTRYPOINT — WR-HBP Final Outer-Test Confirmation V1

**Do not create a new Kaggle run from this filename.** Use `Coffee17_WR_HBP_Recovery_And_Final_V2_Kaggle.ipynb` instead. The V2 entrypoint was added specifically to avoid stale Kaggle notebook copies that still contained the old checkpoint-required preflight.

This file is retained only for repository history; its code is kept aligned with the recovery-capable flow.

In [ ]:
NOTEBOOK_BUILD = "WR-HBP-FINAL-OUTER-CONFIRMATION-V1"
SCIENTIFIC_CODE_COMMIT = "b62963faec8e337359ba4244a3ee5814709bc18f"
DEVELOPMENT_CODE_COMMIT = "01c9212965bc9040ef151204b9404d564f523a0f"
RECOVERY_CODE_COMMIT = "9004462d4c114e594aced9954a56435231c84ac3"
ORIGINAL_SHARED_CORE_SHA256 = "6d425c6c149005afde1224ed74ccfe4eb84c95e574d21e25a0e728578d79f9cc"

import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

import importlib, json, shutil, subprocess, sys, tempfile, zipfile
from pathlib import Path

INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")
REPO = WORK / "coffee-bean-classification-wr-final"
PROJECT = WORK / "wr-hbp-final-confirmation-v1-project"
OUT = PROJECT / "outer"
AUTHORITY = PROJECT / "authority/wr_hbp_final_test_authority.json"
SUMMARY = PROJECT / "analysis/wr_hbp_final_outer_summary.json"

assert INPUT.is_dir() and WORK.is_dir(), "Notebook ini harus dijalankan di Kaggle."

def run(command, cwd=None, log_path=None):
    command = [str(x) for x in command]
    print("\n$ " + " ".join(command), flush=True)
    env = os.environ.copy()
    if log_path is None:
        subprocess.run(command, cwd=cwd, check=True, env=env)
        return
    log_path = Path(log_path)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    with log_path.open("a", encoding="utf-8") as stream:
        process = subprocess.Popen(
            command,
            cwd=cwd,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
            env=env,
        )
        assert process.stdout is not None
        for line in process.stdout:
            print(line, end="")
            stream.write(line)
            stream.flush()
        rc = process.wait()
    if rc:
        raise RuntimeError(f"Command gagal ({rc}): {' '.join(command)}")

# ------------------------------------------------------------------
# 0. Resolve the exact prior WR-HBP development project WITH checkpoints.
#    Accepted Kaggle inputs:
#      (a) saved notebook output directory containing
#          coffee17-wavelet-residual-hbp-project/, or
#      (b) the dedicated final-confirmation checkpoint bundle ZIP produced by
#          the updated WR-HBP development notebook.
#    The compact analysis package is intentionally rejected because it does not
#    contain best.pt/last.pt.
# ------------------------------------------------------------------
DEV_EXPERIMENT_REL = Path("experiments/coffee17-wavelet-residual-hbp-v1")
DEV_EXTRACT = WORK / "_wr_hbp_development_input"

def _required_development_files(project_root):
    project_root = Path(project_root)
    required = []
    for fold in range(1, 6):
        pair = (
            project_root
            / DEV_EXPERIMENT_REL
            / f"fold_{fold}"
            / "seed42"
        )
        required.append(pair / "pair_result.json")
        for arm in ("R0_HBP", "WR_HBP"):
            arm_root = pair / arm
            required.extend([
                arm_root / "best.pt",
                arm_root / "last.pt",
                arm_root / "run_contract.json",
            ])
    return required

def _project_complete(project_root):
    return all(p.is_file() for p in _required_development_files(project_root))

def _zip_project_prefix(archive_path):
    try:
        with zipfile.ZipFile(archive_path) as zf:
            names = {
                name.rstrip("/")
                for name in zf.namelist()
                if name and not name.endswith("/")
            }
    except zipfile.BadZipFile:
        return None

    suffixes = []
    for fold in range(1, 6):
        pair = f"experiments/coffee17-wavelet-residual-hbp-v1/fold_{fold}/seed42"
        suffixes.append(f"{pair}/pair_result.json")
        for arm in ("R0_HBP", "WR_HBP"):
            suffixes.extend([
                f"{pair}/{arm}/best.pt",
                f"{pair}/{arm}/last.pt",
                f"{pair}/{arm}/run_contract.json",
            ])

    candidates = {""}
    marker = suffixes[0]
    for name in names:
        if name.endswith(marker):
            candidates.add(name[:-len(marker)].rstrip("/"))

    valid = []
    for prefix in sorted(candidates):
        def member(suffix):
            return f"{prefix}/{suffix}" if prefix else suffix
        if all(member(suffix) in names for suffix in suffixes):
            valid.append(prefix)

    if len(valid) == 1:
        return valid[0]
    if len(valid) > 1:
        raise RuntimeError(
            f"ZIP development ambigu: {archive_path} memiliki beberapa project lengkap: {valid}"
        )
    return None

directory_candidates = sorted({
    p.resolve()
    for p in INPUT.rglob("coffee17-wavelet-residual-hbp-project")
    if p.is_dir() and _project_complete(p)
})

# Prefer a mounted saved-output directory when present. Only if no complete
# directory exists do we inspect checkpoint bundle ZIPs.
zip_candidates = []
if not directory_candidates:
    for archive in sorted(INPUT.rglob("*.zip")):
        prefix = _zip_project_prefix(archive)
        if prefix is not None:
            zip_candidates.append((archive.resolve(), prefix))

candidate_count = len(directory_candidates) if directory_candidates else len(zip_candidates)
if candidate_count > 1:
    found_dirs = "\n".join(f"  DIR: {p}" for p in directory_candidates) or "  DIR: none"
    found_zips = (
        "\n".join(f"  ZIP: {p} (prefix={prefix or '<root>'})" for p, prefix in zip_candidates)
        or "  ZIP: none"
    )
    raise RuntimeError(
        "Lebih dari satu sumber checkpoint WR-HBP lengkap ditemukan. "
        "Sisakan tepat satu agar provenance tidak ambigu.\n"
        f"{found_dirs}\n{found_zips}"
    )

NEEDS_RECOVERY = candidate_count == 0

if not NEEDS_RECOVERY and directory_candidates:
    DEV_PROJECT = directory_candidates[0]
    DEV_SOURCE = f"directory:{DEV_PROJECT}"
elif not NEEDS_RECOVERY:
    archive, prefix = zip_candidates[0]
    shutil.rmtree(DEV_EXTRACT, ignore_errors=True)
    DEV_EXTRACT.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive) as zf:
        for info in zf.infolist():
            member = Path(info.filename)
            if member.is_absolute() or ".." in member.parts:
                raise RuntimeError(f"Unsafe ZIP member ditolak: {info.filename}")
        zf.extractall(DEV_EXTRACT)
    DEV_PROJECT = DEV_EXTRACT / prefix if prefix else DEV_EXTRACT
    if not _project_complete(DEV_PROJECT):
        raise RuntimeError(
            f"Checkpoint bundle berhasil diekstrak tetapi project tidak lengkap: {archive}"
        )
    DEV_SOURCE = f"zip:{archive}"
else:
    # Pre-test amendment: exact selected checkpoints are unavailable, so one
    # strict-deterministic reconstruction is allowed before any outer-test
    # identity is materialized.
    DEV_PROJECT = WORK / "coffee17-wavelet-residual-hbp-recovery-project"
    shutil.rmtree(DEV_PROJECT, ignore_errors=True)
    DEV_SOURCE = "checkpoint_loss_recovery_v1"

DEV_ROOT = DEV_PROJECT / DEV_EXPERIMENT_REL

if not NEEDS_RECOVERY:
    missing = [
        str(p) for p in _required_development_files(DEV_PROJECT)
        if not p.is_file()
    ]
    if missing:
        raise RuntimeError(
            "Development project tidak lengkap setelah resolver:\n"
            + "\n".join(missing)
        )
    print("DEVELOPMENT SOURCE:", DEV_SOURCE)
    print("DEVELOPMENT PROJECT:", DEV_PROJECT)
    print("DEVELOPMENT CHECKPOINT PREFLIGHT: PASS")
    print("Recovery training: NOT NEEDED")
else:
    analysis_zips = sorted(
        p for p in INPUT.rglob("*.zip")
        if "analysis-package" in p.name.lower()
    )
    print("DEVELOPMENT SOURCE: exact checkpoint project/bundle not found")
    if analysis_zips:
        print("Compact analysis package found but it has no best.pt/last.pt:")
        for p in analysis_zips:
            print(" -", p)
    print("RECOVERY MODE: checkpoint_loss_recovery_v1")
    print("Outer test remains locked until deterministic development recovery passes.")


# ------------------------------------------------------------------
# 1. Checkout the code needed before outer-test access.
#    Original checkpoint path keeps the original final code pin.
#    Recovery path uses the preregistered deterministic recovery commit.
# ------------------------------------------------------------------
PRETEST_CODE_COMMIT = RECOVERY_CODE_COMMIT if NEEDS_RECOVERY else SCIENTIFIC_CODE_COMMIT

if REPO.exists():
    shutil.rmtree(REPO)
run([
    "git", "clone", "--quiet", "--no-checkout",
    "https://github.com/ediprin/coffee-bean-classification.git", REPO
])
run([
    "git", "-C", REPO, "checkout", "--quiet", "--detach",
    PRETEST_CODE_COMMIT
])
checked_out = subprocess.check_output(
    ["git", "-C", REPO, "rev-parse", "HEAD"], text=True
).strip()
if checked_out != PRETEST_CODE_COMMIT:
    raise RuntimeError(f"Checkout salah: {checked_out} != {PRETEST_CODE_COMMIT}")

run([
    sys.executable, "-m", "pip", "install", "-q", "-r",
    REPO / "requirements/preprocessing-study.txt"
])
run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", REPO])
sys.path.insert(0, str(REPO / "src"))
importlib.invalidate_caches()
os.chdir(REPO)

import torch
if not torch.cuda.is_available():
    raise RuntimeError("Aktifkan Kaggle GPU sebelum Run All.")

print("NOTEBOOK_BUILD:", NOTEBOOK_BUILD)
print("FINAL_CODE_COMMIT:", SCIENTIFIC_CODE_COMMIT)
print("ORIGINAL_DEVELOPMENT_CODE_COMMIT:", DEVELOPMENT_CODE_COMMIT)
print("RECOVERY_CODE_COMMIT:", RECOVERY_CODE_COMMIT)
print("PRETEST_CODE_COMMIT:", PRETEST_CODE_COMMIT)
print("GPU:", torch.cuda.get_device_name(0))

pretest_tests = ["tests/experiments/test_wr_hbp_final_confirmation.py"]
if NEEDS_RECOVERY:
    pretest_tests.append("tests/modeling/test_wavelet_residual_hbp.py")
# Do not execute source-inspection tests for the notebook entrypoint from the
# pinned recovery checkout. GitHub CI validates notebook source at branch HEAD;
# Kaggle preflight validates the scientific/runtime code only.
run([
    sys.executable, "-m", "pytest", "-q", *pretest_tests,
    "-k", "not kaggle_notebook",
], cwd=REPO)
print("PRE-TEST SCIENTIFIC CODE CHECKS: PASS")

PROJECT.mkdir(parents=True, exist_ok=True)
(PROJECT / "logs").mkdir(parents=True, exist_ok=True)
(PROJECT / "authority").mkdir(parents=True, exist_ok=True)
(PROJECT / "analysis").mkdir(parents=True, exist_ok=True)
OUT.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------------
# 2. Reconstruct the exact frozen Coffee17 manifest.
#    Still NO outer-test materialization here.
# ------------------------------------------------------------------
from bilinear_lmmd.data.preparation.prepare_coffee17 import discover_directory_samples
from bilinear_lmmd.data.preparation.audit_coffee17_provenance import audit_coffee17_provenance
from bilinear_lmmd.data.preparation.prepare_preprocessing_folds import prepare_preprocessing_folds

if NEEDS_RECOVERY:
    from bilinear_lmmd.data.preparation.materialize_preprocessing_development import (
        materialize_preprocessing_development,
    )

by_class = discover_directory_samples(INPUT)
raw_count = sum(len(v) for v in by_class.values())
print("Coffee17 mounted:", raw_count, "images /", len(by_class), "classes")
if raw_count != 979 or len(by_class) != 17:
    raise RuntimeError(
        "Input harus Coffee17 original: "
        f"expected 979 images / 17 classes, observed {raw_count}/{len(by_class)}."
    )

ARCHIVE = WORK / "coffee17_wr_final_original.zip"
PROV_LOCAL = WORK / "coffee17_wr_final_provenance"
CANONICAL = WORK / "coffee17_wr_final_original_v1"
FOLDS_LOCAL = WORK / "coffee17_wr_final_folds"

for path in (PROV_LOCAL, CANONICAL, FOLDS_LOCAL):
    shutil.rmtree(path, ignore_errors=True)
if ARCHIVE.exists():
    ARCHIVE.unlink()

with zipfile.ZipFile(ARCHIVE, "w", compression=zipfile.ZIP_STORED) as bundle:
    for class_name, paths in sorted(by_class.items()):
        for path in sorted(paths):
            info = zipfile.ZipInfo(
                f"{class_name}/{path.name}",
                date_time=(1980, 1, 1, 0, 0, 0),
            )
            info.compress_type = zipfile.ZIP_STORED
            info.external_attr = 0o644 << 16
            bundle.writestr(info, path.read_bytes())

provenance = audit_coffee17_provenance(
    ARCHIVE, PROV_LOCAL, canonical_root=CANONICAL
)
if provenance["decision"] != "PASS":
    raise RuntimeError(f"Provenance gagal: {provenance['decision']}")

fold_summary = prepare_preprocessing_folds(
    CANONICAL,
    PROV_LOCAL / "coffee17_provenance.json",
    FOLDS_LOCAL,
    folds=5,
    seed=42,
    validation_ratio=0.10,
)
if fold_summary["decision"] != "PASS_COFFEE17_PREPROCESSING_DATA_GATE":
    raise RuntimeError("Frozen fold reconstruction gagal.")

CLEAN = FOLDS_LOCAL / "clean_manifest.json"
FOLD_MANIFEST = FOLDS_LOCAL / "fold_manifest.json"

# ------------------------------------------------------------------
# 2b. If exact checkpoints were lost, reconstruct them ONCE under the
#     preregistered strict-deterministic recovery amendment.
# ------------------------------------------------------------------
if NEEDS_RECOVERY:
    RECOVERY_CONFIG = REPO / "configs/wavelet_residual_hbp/WR_HBP_V1.yaml"
    DEV_ROOT.mkdir(parents=True, exist_ok=True)

    from bilinear_lmmd.core.config import load_config as _load_recovery_config
    from bilinear_lmmd.engine.wavelet_residual_hbp import (
        preflight_matched_initialization as _wr_preflight,
    )

    _recovery_cfg = _load_recovery_config(RECOVERY_CONFIG)
    _preflight = _wr_preflight(_recovery_cfg)
    if _preflight["shared_core_sha256"] != ORIGINAL_SHARED_CORE_SHA256:
        raise RuntimeError(
            "Recovery shared-core fingerprint berbeda dari WR-HBP V1 original: "
            f"{_preflight['shared_core_sha256']} != {ORIGINAL_SHARED_CORE_SHA256}"
        )
    print("ORIGINAL SHARED-CORE FINGERPRINT: PASS", ORIGINAL_SHARED_CORE_SHA256)

    import importlib.metadata as _pkgmeta
    _packages = [
        "torch", "torchvision", "timm", "numpy", "scikit-learn",
        "scikit-image", "PyWavelets", "Pillow", "scipy",
    ]
    _environment = {
        "recovery_code_commit": RECOVERY_CODE_COMMIT,
        "original_shared_core_sha256": ORIGINAL_SHARED_CORE_SHA256,
        "observed_shared_core_sha256": _preflight["shared_core_sha256"],
        "python": sys.version,
        "packages": {},
    }
    for _name in _packages:
        try:
            _environment["packages"][_name] = _pkgmeta.version(_name)
        except _pkgmeta.PackageNotFoundError:
            _environment["packages"][_name] = None
    (DEV_PROJECT / "recovery_environment.json").write_text(
        json.dumps(_environment, indent=2) + "\n",
        encoding="utf-8",
    )

    print("\n=== CHECKPOINT-LOSS RECOVERY: 5 MATCHED DEVELOPMENT FOLDS ===")
    for fold in range(1, 6):
        print(f"\n============ RECOVERY FOLD {fold}/5 ============", flush=True)
        DEV_RUNTIME = WORK / f"coffee17_wr_recovery_dev_fold_{fold}"
        shutil.rmtree(DEV_RUNTIME, ignore_errors=True)

        materialize_preprocessing_development(
            CANONICAL,
            CLEAN,
            FOLD_MANIFEST,
            DEV_RUNTIME,
            fold=fold,
        )

        run([
            sys.executable, "-u", "-m",
            "bilinear_lmmd.experiments.run_wavelet_residual_hbp_matched",
            "--config", RECOVERY_CONFIG,
            "--fold", str(fold),
            "--data-root", DEV_RUNTIME,
            "--output-root", DEV_ROOT,
            "--required-commit", RECOVERY_CODE_COMMIT,
            "--device", "cuda:0",
            "--resume",
            "--authorize-training",
            "--strict-determinism",
        ], cwd=REPO, log_path=PROJECT / "logs" / f"recovery_fold_{fold}.log")

        pair_result = json.loads(
            (DEV_ROOT / f"fold_{fold}" / "seed42" / "pair_result.json")
            .read_text(encoding="utf-8")
        )
        d = pair_result["DELTA_WR_MINUS_R0"]
        print(
            f"RECOVERY FOLD {fold} | "
            f"Delta Macro={d['macro_f1']:+.4f} | "
            f"Delta Hard={d['hard_class_f1']:+.4f} | "
            f"Delta Worst={d['worst_class_f1']:+.4f}"
        )
        shutil.rmtree(DEV_RUNTIME, ignore_errors=True)
        torch.cuda.empty_cache()

    if not _project_complete(DEV_PROJECT):
        missing = [
            str(p) for p in _required_development_files(DEV_PROJECT)
            if not p.is_file()
        ]
        raise RuntimeError(
            "Recovery selesai tetapi checkpoint project tidak lengkap:\n"
            + "\n".join(missing)
        )
    print("\nDETERMINISTIC CHECKPOINT RECOVERY COMPLETE.")
    print("Outer test is still untouched.")
# ------------------------------------------------------------------
# 3. PRE-TEST AUTHORITY.
#    This reads ONLY development artifacts + manifests.
#    If it fails, test images are never materialized.
# ------------------------------------------------------------------
if AUTHORITY.exists():
    AUTHORITY.unlink()

authority_cmd = [
    sys.executable, "-u", "-m",
    "bilinear_lmmd.experiments.build_wr_hbp_final_test_authority",
    "--development-root", DEV_ROOT,
    "--clean-manifest", CLEAN,
    "--fold-manifest", FOLD_MANIFEST,
    "--output", AUTHORITY,
]
if NEEDS_RECOVERY:
    authority_cmd.extend([
        "--recovery-development-commit", RECOVERY_CODE_COMMIT,
    ])

run(
    authority_cmd,
    cwd=REPO,
    log_path=PROJECT / "logs/authority.log",
)

authority = json.loads(AUTHORITY.read_text(encoding="utf-8"))
if authority["decision"] != "AUTHORIZE_OOF_TEST_EVALUATION":
    raise RuntimeError("Authority tidak mengizinkan outer test.")
if authority["test_images_accessed"] is not False:
    raise RuntimeError("Authority dibangun setelah test access — STOP.")
if authority["further_primary_tuning_authorized"] is not False:
    raise RuntimeError("Authority masih mengizinkan tuning — STOP.")

expected_development_commit = (
    RECOVERY_CODE_COMMIT if NEEDS_RECOVERY else DEVELOPMENT_CODE_COMMIT
)
if authority["development_scientific_commit"] != expected_development_commit:
    raise RuntimeError(
        "Development scientific commit tidak cocok: "
        f"{authority['development_scientific_commit']} != "
        f"{expected_development_commit}"
    )
expected_mode = (
    "checkpoint_loss_recovery_v1"
    if NEEDS_RECOVERY
    else "exact_original_checkpoints"
)
if authority.get("development_mode", "exact_original_checkpoints") != expected_mode:
    raise RuntimeError(
        "Development authority mode tidak cocok: "
        f"{authority.get('development_mode')} != {expected_mode}"
    )

print("\n=== PRE-TEST AUTHORITY PASS ===")
print("Development mode:", expected_mode)
print("Development gate:", authority["development_gate"])
print("Clean count:", authority["clean_count"])
print("Further tuning authorized:", authority["further_primary_tuning_authorized"])

# Persist recovered checkpoints BEFORE any outer-test materialization.
if NEEDS_RECOVERY:
    RECOVERY_BUNDLE_STAGE = WORK / "_wr_hbp_recovery_bundle_stage"
    shutil.rmtree(RECOVERY_BUNDLE_STAGE, ignore_errors=True)
    bundle_project = RECOVERY_BUNDLE_STAGE / "coffee17-wavelet-residual-hbp-project"
    shutil.copytree(DEV_PROJECT, bundle_project)
    recovery_bundle = Path(
        shutil.make_archive(
            str(WORK / "wavelet-residual-hbp-recovery-checkpoint-bundle"),
            "zip",
            root_dir=RECOVERY_BUNDLE_STAGE,
        )
    )
    print("RECOVERY CHECKPOINT BUNDLE:", recovery_bundle)
    print(
        "RECOVERY BUNDLE SIZE:",
        round(recovery_bundle.stat().st_size / 1024 / 1024, 2),
        "MB",
    )
    shutil.rmtree(RECOVERY_BUNDLE_STAGE, ignore_errors=True)

AMENDED_PROTOCOL_COPY = PROJECT / "authority/WR_HBP_FINAL_OUTER_CONFIRMATION_V1_AMENDED.md"
if NEEDS_RECOVERY:
    shutil.copy2(
        REPO / "docs/protocols/WR_HBP_FINAL_OUTER_CONFIRMATION_V1.md",
        AMENDED_PROTOCOL_COPY,
    )

# Switch back to the ORIGINAL frozen final-inference commit before any test
# identity is materialized. Recovery code is never used for outer inference.
if PRETEST_CODE_COMMIT != SCIENTIFIC_CODE_COMMIT:
    run([
        "git", "-C", REPO, "checkout", "--quiet", "--detach",
        SCIENTIFIC_CODE_COMMIT
    ])
    final_checked_out = subprocess.check_output(
        ["git", "-C", REPO, "rev-parse", "HEAD"], text=True
    ).strip()
    if final_checked_out != SCIENTIFIC_CODE_COMMIT:
        raise RuntimeError(
            f"Final checkout salah: {final_checked_out} != {SCIENTIFIC_CODE_COMMIT}"
        )
    run([
        sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", REPO
    ])
    run([
        sys.executable, "-m", "pytest", "-q",
        "tests/experiments/test_wr_hbp_final_confirmation.py",
    ], cwd=REPO)
    print("FINAL INFERENCE CODE RESTORED:", SCIENTIFIC_CODE_COMMIT)

CONFIG = REPO / "configs/wr_hbp_final_confirmation/WR_HBP_FINAL_CONFIRMATION_V1.yaml"
# ------------------------------------------------------------------
# 4. ONE-SHOT outer test: materialize one fold, infer both frozen arms,
#    then destroy that temporary test runtime.
# ------------------------------------------------------------------
for fold in range(1, 6):
    print(f"\n================ OUTER FOLD {fold}/5 ================", flush=True)
    with tempfile.TemporaryDirectory(
        prefix=f"coffee17_wr_final_test_fold{fold}_", dir=WORK
    ) as tmp:
        TEST_ROOT = Path(tmp)

        run([
            sys.executable, "-u", "-m",
            "bilinear_lmmd.data.preparation.materialize_preprocessing_test",
            "--canonical-root", CANONICAL,
            "--clean-manifest", CLEAN,
            "--fold-manifest", FOLD_MANIFEST,
            "--authority", AUTHORITY,
            "--destination", TEST_ROOT,
            "--fold", str(fold),
            "--authorize-test",
        ], cwd=REPO)
        contract = json.loads(
            (TEST_ROOT / "test_contract.json").read_text(encoding="utf-8")
        )
        if contract["training_executed"] is not False:
            raise RuntimeError("Test contract unexpectedly indicates training.")

        run([
            sys.executable, "-u", "-m",
            "bilinear_lmmd.experiments.run_wr_hbp_final_outer_fold",
            "--config", CONFIG,
            "--fold", str(fold),
            "--test-root", TEST_ROOT,
            "--development-root", DEV_ROOT,
            "--authority", AUTHORITY,
            "--output-root", OUT,
            "--required-commit", SCIENTIFIC_CODE_COMMIT,
            "--device", "cuda:0",
        ], cwd=REPO, log_path=PROJECT / "logs" / f"outer_fold_{fold}.log")

    result = json.loads(
        (OUT / f"fold_{fold}" / "fold_result.json").read_text(encoding="utf-8")
    )
    d = result["DELTA_WR_MINUS_R0"]
    print(
        f"OUTER FOLD {fold} | "
        f"R0 Macro={result['R0_HBP']['macro_f1']:.4f} | "
        f"WR Macro={result['WR_HBP']['macro_f1']:.4f} | "
        f"Delta Macro={d['macro_f1']:+.4f} | "
        f"Delta Hard={d['hard_class_f1']:+.4f} | "
        f"Delta Worst={d['worst_class_f1']:+.4f}"
    )
    torch.cuda.empty_cache()

# ------------------------------------------------------------------
# 5. Pooled OOF summary + frozen gate + 10,000 paired stratified bootstrap.
# ------------------------------------------------------------------
run([
    sys.executable, "-u", "-m",
    "bilinear_lmmd.experiments.run_wr_hbp_final_outer_summary",
    "--output-root", OUT,
    "--authority", AUTHORITY,
    "--config", CONFIG,
    "--clean-manifest", CLEAN,
    "--output", SUMMARY,
], cwd=REPO, log_path=PROJECT / "logs/summary.log")

summary = json.loads(SUMMARY.read_text(encoding="utf-8"))
print("\n=== FINAL CONFIRMATION ===")
print("Decision:", summary["confirmation_gate"]["decision"])
print("OOF identities:", summary["oof_identity_count"])
print("R0 Macro-F1:", f"{summary['pooled']['R0_HBP']['macro_f1']:.4%}")
print("WR Macro-F1:", f"{summary['pooled']['WR_HBP']['macro_f1']:.4%}")
print("Delta Macro-F1:", f"{summary['delta_wr_minus_r0']['macro_f1']:+.4%}")
print("Delta Hard-F1:", f"{summary['delta_wr_minus_r0']['hard_class_f1']:+.4%}")
print("Delta Worst-F1:", f"{summary['delta_wr_minus_r0']['worst_class_f1']:+.4%}")
print("Positive Macro outer folds:", summary["positive_macro_outer_folds"], "/5")
print(
    "Rescue / damage:",
    summary["paired_prediction_outcomes"]["rescue"],
    "/",
    summary["paired_prediction_outcomes"]["damage"],
)
boot = summary["paired_stratified_bootstrap_macro_delta"]
print(
    "Bootstrap Macro delta 95% CI:",
    f"[{boot['lower_95']:+.4%}, {boot['upper_95']:+.4%}]",
    "| P(delta>0)=", f"{boot['probability_delta_gt_zero']:.3f}",
)
print("Further tuning authorized:", summary["further_tuning_authorized"])

# ------------------------------------------------------------------
# 6. Compact result package. No checkpoints are modified or retrained.
# ------------------------------------------------------------------
PKG = WORK / "wr-hbp-final-confirmation-v1-analysis-package"
shutil.rmtree(PKG, ignore_errors=True)
PKG.mkdir(parents=True)

shutil.copy2(AUTHORITY, PKG / AUTHORITY.name)
shutil.copy2(SUMMARY, PKG / SUMMARY.name)
shutil.copy2(CONFIG, PKG / CONFIG.name)
if NEEDS_RECOVERY and (DEV_PROJECT / "recovery_environment.json").is_file():
    shutil.copy2(
        DEV_PROJECT / "recovery_environment.json",
        PKG / "recovery_environment.json",
    )
protocol_source = (
    AMENDED_PROTOCOL_COPY
    if NEEDS_RECOVERY and AMENDED_PROTOCOL_COPY.is_file()
    else REPO / "docs/protocols/WR_HBP_FINAL_OUTER_CONFIRMATION_V1.md"
)
shutil.copy2(
    protocol_source,
    PKG / "WR_HBP_FINAL_OUTER_CONFIRMATION_V1.md",
)

for fold in range(1, 6):
    src = OUT / f"fold_{fold}"
    dst = PKG / f"fold_{fold}"
    dst.mkdir(parents=True)
    shutil.copy2(src / "fold_result.json", dst / "fold_result.json")
    for arm in ("R0_HBP", "WR_HBP"):
        arm_src = src / arm
        arm_dst = dst / arm
        arm_dst.mkdir(parents=True)
        for name in ("metrics.json", "predictions.csv"):
            shutil.copy2(arm_src / name, arm_dst / name)

zip_path = Path(shutil.make_archive(
    str(WORK / "wr-hbp-final-confirmation-v1-analysis-package"),
    "zip",
    root_dir=PKG,
))
print("\nREADY:", zip_path)
print("SIZE:", round(zip_path.stat().st_size / 1024 / 1024, 2), "MB")
print("FINAL PROTOCOL CLOSED: no post-test tuning is authorized.")

# Remove temporary reconstruction/code only. Keep PROJECT + result ZIP.
for path in (REPO, PROV_LOCAL, CANONICAL, FOLDS_LOCAL, PKG):
    shutil.rmtree(path, ignore_errors=True)
if ARCHIVE.exists():
    ARCHIVE.unlink()
